# Notebook 01: Ingesta y Estandarización de Datos (ENCSPA 2019)

**Objetivos de este notebook:**
1. Leer todos los archivos CSV originales (`raw`) proporcionados por el DANE.
2. Estandarizar los nombres de las columnas (minúsculas, sin espacios, sin caracteres especiales) para evitar errores tipográficos en el código.
3. Hacer una revisión rápida de duplicados y valores nulos estructurales.
4. Exportar los datos limpios en formato `.parquet` a la carpeta `interim` para que la lectura futura sea más rápida y consuma menos memoria RAM.

**Nota metodológica:** En este paso NO vamos a reemplazar los números "9", "99" o "999" por nulos (`NaN`), ya que dependiendo de la variable, un "9" puede significar "No sabe/No responde" o puede ser una cantidad válida (ej. consumió 9 cigarrillos). El reemplazo semántico se hará en el Notebook 02.

In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# Configurar pandas para mostrar todas las columnas al imprimir
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [4]:
# Definir las rutas de la estructura del proyecto
BASE_DIR = Path('..') # Sube un nivel a la raíz del proyecto
RAW_DIR = BASE_DIR / 'data' / 'raw'
INTERIM_DIR = BASE_DIR / 'data' / 'interim'

# Crear la carpeta interim si no existe
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Buscar todos los archivos CSV en la carpeta raw
csv_files = glob.glob(str(RAW_DIR / '*.csv'))

if len(csv_files) == 0:
    print(f"ADVERTENCIA: No se encontraron archivos CSV en {RAW_DIR}.")
    print("Asegúrate de haber guardado los CSV del DANE en la carpeta data/raw/")
else:
    print(f"Se encontraron {len(csv_files)} archivos CSV listos para procesar.")

Se encontraron 20 archivos CSV listos para procesar.


In [5]:
def clean_column_names(df):
    """
    Función para estandarizar los nombres de las columnas de un DataFrame.
    Pasa todo a minúsculas, quita espacios en blanco al inicio/final 
    y reemplaza espacios internos por guiones bajos.
    """
    df.columns = (
        df.columns
        .str.strip()               # Quitar espacios al inicio y final
        .str.lower()               # Pasar a minúsculas
        .str.replace(' ', '_')     # Reemplazar espacios por guiones bajos
        .str.replace('[^a-z0-9_]', '', regex=True) # Quitar caracteres raros
    )
    return df

def read_dane_csv(file_path):
    """
    Función robusta para leer los CSV del DANE. 
    Intenta leer con separador ';' y si falla intenta con ','.
    Maneja problemas de codificación (UTF-8 vs Latin-1).
    """
    try:
        # Los archivos del DANE suelen venir separados por punto y coma y en latin1 o utf-8
        df = pd.read_csv(file_path, sep=';', encoding='utf-8', low_memory=False)
        if df.shape[1] == 1: # Si solo leyó una columna, el separador era coma
            df = pd.read_csv(file_path, sep=',', encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, sep=';', encoding='latin-1', low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(file_path, sep=',', encoding='latin-1', low_memory=False)
            
    return df

In [6]:
# Diccionario para almacenar información rápida del proceso
etl_summary = []

print("Iniciando procesamiento de archivos...\n" + "-"*40)

for file in csv_files:
    file_name = os.path.basename(file)
    table_name = file_name.replace('.csv', '')
    
    print(f"Procesando: {file_name}...")
    
    # 1. Leer el CSV
    df = read_dane_csv(file)
    initial_shape = df.shape
    
    # 2. Estandarizar columnas
    df = clean_column_names(df)
    
    # 3. Eliminar filas que sean 100% nulas o duplicados exactos (limpieza estructural)
    df = df.dropna(how='all')
    df = df.drop_duplicates()
    final_shape = df.shape
    
    # 4. Guardar en formato Parquet en la carpeta interim
    parquet_path = INTERIM_DIR / f"{table_name}.parquet"
    df.to_parquet(parquet_path, index=False)
    
    # Guardar métricas
    etl_summary.append({
        'Archivo': file_name,
        'Filas Originales': initial_shape[0],
        'Columnas': initial_shape[1],
        'Filas Limpias': final_shape[0],
        'Duplicados Removidos': initial_shape[0] - final_shape[0]
    })

print("-"*40 + "\n Proceso finalizado. Todos los archivos han sido guardados en .parquet")

Iniciando procesamiento de archivos...
----------------------------------------
Procesando: d2_capitulos.csv...
Procesando: d_capitulos.csv...
Procesando: encuestas.csv...
Procesando: e_capitulos.csv...
Procesando: f_capitulos.csv...
Procesando: g_capitulos.csv...
Procesando: h_capitulos.csv...
Procesando: i_capitulos.csv...
Procesando: j_capitulos.csv...
Procesando: k_capitulos.csv...
Procesando: l_capitulos.csv...
Procesando: m_capitulos.csv...
Procesando: n_capitulos.csv...
Procesando: o_capitulos.csv...
Procesando: personas.csv...
Procesando: personas_seleccionadas.csv...
Procesando: p_capitulos.csv...
Procesando: q_capitulos.csv...
Procesando: r_capitulos.csv...
Procesando: s_capitulos.csv...
----------------------------------------
 Proceso finalizado. Todos los archivos han sido guardados en .parquet


In [7]:
# Mostrar el resumen de lo que acabamos de hacer
df_summary = pd.DataFrame(etl_summary)
df_summary

,Archivo,Filas Originales,Columnas,Filas Limpias,Duplicados Removidos
0,d2_capitulos.csv,49756,11,49756,0
1,d_capitulos.csv,49756,34,49756,0
2,encuestas.csv,49760,11,49760,0
3,e_capitulos.csv,49756,16,49756,0
4,f_capitulos.csv,49756,37,49756,0
5,g_capitulos.csv,49756,98,49756,0
6,h_capitulos.csv,909,34,909,0
7,i_capitulos.csv,63,26,63,0
8,j_capitulos.csv,780,29,780,0
9,k_capitulos.csv,3982,48,3982,0


In [8]:
# Escojamos un archivo clave, por ejemplo 'personas_seleccionadas', para verificar cómo quedó
try:
    path_prueba = INTERIM_DIR / 'personas_seleccionadas.parquet'
    df_prueba = pd.read_parquet(path_prueba)
    
    print("Muestra de datos de 'personas_seleccionadas':")
    display(df_prueba.head())
    
    print("\nTipos de datos de las columnas identificadoras:")
    # Verificar que las llaves de cruce existan y ver sus tipos
    llaves = ['directorio', 'secuencia_encuesta', 'secuencia_p', 'orden']
    display(df_prueba[llaves].dtypes)
    
except Exception as e:
    print("Asegúrate de que el archivo personas_seleccionadas existía en tu carpeta raw.", e)

Muestra de datos de 'personas_seleccionadas':


,sexo,edad,parentesco,padre,madre,consentimiento,resultado,directorio,secuencia_encuesta,secuencia_p,orden,fex_c
0,2,56,2,2,2,1,1,1,2,1,2,1047.148485
1,2,62,1,2,2,1,1,2,1,1,1,478.696450
2,1,54,1,3,2,1,1,3,1,1,1,460.912269
3,2,64,1,3,3,1,1,4,1,1,1,239.348225
4,2,46,1,2,2,1,1,6,1,1,1,203.978981



Tipos de datos de las columnas identificadoras:


directorio            int64
secuencia_encuesta    int64
secuencia_p           int64
orden                 int64
dtype: object